<a href="https://colab.research.google.com/github/Jirtus-sanasam/MLP-Diabetes/blob/main/Stacking1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np

In [15]:
df = pd.read_csv('/content/diabetes_data2.csv')

In [16]:
# Replacing 0 as null value
cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols] = df[cols].replace(0, np.nan)

In [17]:
# Imputing Null Value with KNNImputer
from sklearn.impute import KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)
df[cols] = knn_imputer.fit_transform(df[cols])

In [18]:
# Cap outliers using IQR
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    return df

for col in df.columns[:-1]:  # exclude target
    df = cap_outliers(df, col)

In [19]:
# FEATURE ENGINEERING

# --- 1. Interaction Features ---
df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Glucose_Insulin'] = df['Glucose'] * df['Insulin']
df['BMI_Age'] = df['BMI'] * df['Age']
df['Insulin_SkinThickness'] = df['Insulin'] * df['SkinThickness']

# --- 2. Ratio Features (avoid division by zero) ---
df['Glucose_Insulin_Ratio'] = df['Glucose'] / (df['Insulin'] + 1e-6)
df['BMI_Age_Ratio'] = df['BMI'] / (df['Age'] + 1e-6)

# --- 3. Polynomial Features ---
df['Glucose_sq'] = df['Glucose'] ** 2
df['BMI_sq'] = df['BMI'] ** 2
df['Age_sq'] = df['Age'] ** 2

# --- 4. Age Binning ---
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[0, 30, 45, 60, 100],
    labels=['Young', 'Middle', 'Senior', 'Elderly']
)
# One-hot encoding with integer output
df = pd.get_dummies(df, columns=['Age_Group'], drop_first=True, dtype=int)

# --- 5. BMI Category ---
df['BMI_Category'] = pd.cut(
    df['BMI'],
    bins=[0, 18.5, 24.9, 29.9, 100],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

df = pd.get_dummies(df, columns=['BMI_Category'], drop_first=True, dtype=int)
# --- 6. Glucose Category ---
df['Glucose_Level'] = pd.cut(
    df['Glucose'],
    bins=[0, 99, 125, 300],
    labels=['Normal', 'Prediabetes', 'Diabetes']
)

df = pd.get_dummies(df, columns=['Glucose_Level'], drop_first=True, dtype=int)

# --- 7. Insulin Resistance (HOMA-IR) ---
df['HOMA_IR'] = (df['Glucose'] * df['Insulin']) / 405
# --- 8. Pregnancy Risk Flag ---
df['High_Pregnancies'] = (df['Pregnancies'] > 3).astype(int)

# --- 9. Log Transform (handle skewness) ---
df['Log_Insulin'] = np.log1p(df['Insulin'])
df['Log_DiabetesPedigreeFunction'] = np.log1p(df['DiabetesPedigreeFunction'])

# --- 10. Composite Risk Score ---
df['Risk_Score'] = (
    (df['Glucose'] / 100) +
    (df['BMI'] / 25) +
    (df['Age'] / 50) +
    (df['DiabetesPedigreeFunction'])
)

# --- 11. Final Safety Check (convert any leftover bool → int) ---
bool_cols = df.select_dtypes(include='bool').columns
if not bool_cols.empty:
    for col in bool_cols.unique(): # Iterate over unique column names
        # Ensure we are only trying to convert actual boolean columns that might still exist
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

In [20]:
X = df.drop('Outcome', axis=1)
Y = df['Outcome']

# Use only RFECV selected features
selected_features = ['Glucose', 'Glucose_BMI', 'Glucose_Insulin', 'BMI_Age',
                     'Insulin_SkinThickness', 'BMI_Age_Ratio', 'Glucose_sq',
                     'BMI_sq', 'HOMA_IR', 'Risk_Score']


In [21]:
# Select only important features
X = X[selected_features]

# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [22]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", dict(pd.Series(y_train).value_counts()))
print("After SMOTE: ", dict(pd.Series(y_train_sm).value_counts()))

Before SMOTE: {0: np.int64(400), 1: np.int64(214)}
After SMOTE:  {0: np.int64(400), 1: np.int64(400)}


In [28]:
# Install LightGBM if not installed
# !pip install lightgbm

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Initialize model
lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0:1, 1:2},
    random_state=42
)

In [30]:
# Stratified K-Fold (maintains class balance)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    lgbm_model,
    X_train,
    y_train,
    cv=skf,
    scoring='accuracy'
)

print("Cross Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

[LightGBM] [Info] Number of positive: 171, number of negative: 320
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000070 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1455
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.516616 -> initscore=0.066490
[LightGBM] [Info] Start training from score 0.066490
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

In [31]:
lgbm_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 214, number of negative: 400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000081 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1772
[LightGBM] [Info] Number of data points in the train set: 614, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.516908 -> initscore=0.067659
[LightGBM] [Info] Start training from score 0.067659
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

LGBMClassifier(class_weight={0: 1, 1: 2}, colsample_bytree=0.8,
               learning_rate=0.03, max_depth=6, n_estimators=500,
               random_state=42, subsample=0.8)

In [32]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = lgbm_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test Accuracy: 0.7142857142857143

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.76      0.78       100
           1       0.59      0.63      0.61        54

    accuracy                           0.71       154
   macro avg       0.69      0.69      0.69       154
weighted avg       0.72      0.71      0.72       154



In [33]:
from sklearn.svm import SVC

In [34]:
svm_model = SVC(
    kernel='rbf',          # best default kernel
    C=1.0,
    gamma='scale',
    class_weight={0:1, 1:2},  # 🔥 important for diabetic class
    probability=True,      # needed for threshold tuning
    random_state=42
)

svm_model.fit(X_train, y_train)

SVC(class_weight={0: 1, 1: 2}, probability=True, random_state=42)

In [35]:
from sklearn.model_selection import cross_val_score

svm_cv_scores = cross_val_score(
    svm_model,
    X_train,
    y_train,
    cv=skf,
    scoring='recall'   # 🔥 prioritize diabetic detection
)

print("SVM CV Recall Scores:", svm_cv_scores)
print("Mean CV Recall:", svm_cv_scores.mean())

SVM CV Recall Scores: [0.69767442 0.72093023 0.76744186 0.76744186 0.71428571]
Mean CV Recall: 0.7335548172757476


In [36]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_svm = svm_model.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.6883116883116883

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.69      0.74       100
           1       0.54      0.69      0.61        54

    accuracy                           0.69       154
   macro avg       0.67      0.69      0.67       154
weighted avg       0.71      0.69      0.69       154



In [37]:
y_probs_svm = svm_model.predict_proba(X_test)[:, 1]

threshold = 0.4  # try 0.3–0.5

y_pred_svm_custom = (y_probs_svm >= threshold).astype(int)

print("\nAfter Threshold Tuning:\n")
print(classification_report(y_test, y_pred_svm_custom))


After Threshold Tuning:

              precision    recall  f1-score   support

           0       0.79      0.72      0.75       100
           1       0.56      0.65      0.60        54

    accuracy                           0.69       154
   macro avg       0.67      0.68      0.68       154
weighted avg       0.71      0.69      0.70       154



In [38]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight={0:1, 1:2},
        probability=True,
        random_state=42
    ))
])

svm_pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svm',
                 SVC(class_weight={0: 1, 1: 2}, probability=True,
                     random_state=42))])

In [39]:
y_pred_pipe = svm_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_pipe))

              precision    recall  f1-score   support

           0       0.88      0.72      0.79       100
           1       0.61      0.81      0.70        54

    accuracy                           0.75       154
   macro avg       0.74      0.77      0.74       154
weighted avg       0.78      0.75      0.76       154



In [40]:
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier

from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [50]:
mlp_model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=1000, # Increased max_iter to allow more time for convergence
        random_state=42
    ))
])

In [51]:
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight={0:1, 1:2},
        probability=True,
        random_state=42
    ))
])

In [52]:
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    class_weight={0:1, 1:2},
    random_state=42
)

In [53]:
stacking_model = StackingClassifier(
    estimators=[
        ('mlp', mlp_model),
        ('svm', svm_model),
        ('lgbm', lgbm_model)
    ],

    final_estimator=LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        class_weight={0:1, 1:2},
        random_state=42
    ),
    cv=5,
    passthrough=True
)

stacking_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 214, number of negative: 400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000077 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1772
[LightGBM] [Info] Number of data points in the train set: 614, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.516908 -> initscore=0.067659
[LightGBM] [Info] Start training from score 0.067659
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perce

[LightGBM] [Info] Number of positive: 171, number of negative: 320
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1451
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.516616 -> initscore=0.066490
[LightGBM] [Info] Start training from score 0.066490
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

StackingClassifier(cv=5,
                   estimators=[('mlp',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('mlp',
                                                 MLPClassifier(hidden_layer_sizes=(64,
                                                                                   32),
                                                               max_iter=1000,
                                                               random_state=42))])),
                               ('svm',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('svm',
                                                 SVC(class_weight={0: 1, 1: 2},
                                                     probability=True,
                                                     random_state=42))])),
                               ('lgbm',
                                LGBMClassifier(class_weight={0: 1, 1: 2},
                                               learning_rate=0.05, max_depth=5,
                                               n_estimators=300,
                                               random_state=42))],
                   final_estimator=LGBMClassifier(class_weight={0: 1, 1: 2},
                                                  learning_rate=0.05,
                                                  max_depth=4, n_estimators=200,
                                                  random_state=42),
                   passthrough=True)

In [55]:
y_pred_stack = stacking_model.predict(X_test)

print("Stacking Ensemble:\n")
print(classification_report(y_test, y_pred_stack))

Stacking Ensemble:

              precision    recall  f1-score   support

           0       0.82      0.79      0.81       100
           1       0.64      0.69      0.66        54

    accuracy                           0.75       154
   macro avg       0.73      0.74      0.73       154
weighted avg       0.76      0.75      0.76       154



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [56]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Define a new Logistic Regression model for the final estimator to allow tuning its parameters
# Note: The StackingClassifier's final_estimator uses clone() internally, so directly defining
# it here and passing its parameters to param_grid is the correct approach.
final_lr = LogisticRegression(random_state=42, solver='liblinear') # Add solver for penalty options

param_grid = {
    'final_estimator__C': [0.1, 1.0, 10.0],
    'final_estimator__penalty': ['l1', 'l2']
}

# Re-instantiate stacking_model with the final_lr estimator to ensure consistency
# If the intent was to tune a LightGBM as final_estimator, then the final_estimator
# in stacking_model would need to be changed to LGBMClassifier, and param_grid would be valid.
# Assuming the user intended to tune LogisticRegression.
stacking_model = StackingClassifier(
    estimators=[
        ('mlp', mlp_model),
        ('svm', svm_model),
        ('lgbm', lgbm_model)
    ],
    final_estimator=final_lr, # Use the LogisticRegression for parameter tuning
    cv=5
)

grid = GridSearchCV(
    stacking_model,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train_sm, y_train_sm)

best_model = grid.best_estimator_

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


[LightGBM] [Info] Number of positive: 400, number of negative: 400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000151 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2524
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.666667 -> initscore=0.693147
[LightGBM] [Info] Start training from score 0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


[LightGBM] [Info] Number of positive: 320, number of negative: 320
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000084 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2129
[LightGBM] [Info] Number of data points in the train set: 640, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.666667 -> initscore=0.693147
[LightGBM] [Info] Start training from score 0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 